## 多输入通道和多输出通道

在前面的学习中，我们的输入数据通常只是一个 $n_h,n_w$ 的矩阵，但是图片一般是彩色的，以常见的 RGB 图片为例，它应该有三层图片组成。所以输入的张量应该是一个形状为 $(c, n_h, n_w)$ 的三维张量，其中 $c$ 在这里，$c = 3$。


## 多输入通道
输入的矩阵的层数，就叫做输入通道数。

对于多输入通道，我们应该有一个和输入通道数相同的卷积核，每个卷积核负责处理一个输入通道。
计算方式如下图:
![两个输入通道互相关计算。](../img/conv-multi-in.svg)

实现如下:

In [13]:
import torch
from d2l import torch as d2l

In [14]:
def corr2d_multi_in(X,K):
    """多输入通道的互相关计算"""
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))


In [15]:
X = torch.tensor([[[1.0, 2.0], [3.0, 4.0],[5.0, 6.0]], [[2.0, 1.0], [4.0, 3.0], [6.0, 5.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 0.0], [3.0, 2.0]]])

print(corr2d_multi_in(X, K))


tensor([[40.],
        [64.]])


## 多输出通道
到目前为止，不论有多少输入通道，我们还只有一个输出通道。然而，每一层有多个输出通道是至关重要的。在最流行的神经网络架构中，随着神经网络层数的加深，我们常会增加输出通道的维数，通过减少空间分辨率以获得更大的通道深度。直观地说，我们可以将每个通道看作对不同特征的响应。而现实可能更为复杂一些，因为每个通道不是独立学习的，而是为了共同使用而优化的。因此，多输出通道并不仅是学习多个单通道的检测器。

用$c_i$和$c_o$分别表示输入和输出通道的数目，并让$k_h$和$k_w$为卷积核的高度和宽度。为了获得多个通道的输出，我们可以为每个输出通道创建一个形状为$c_i\times k_h\times k_w$的卷积核张量，这样卷积核的形状是$c_o\times c_i\times k_h\times k_w$。在互相关运算中，每个输出通道先获取所有输入通道，再以对应该输出通道的卷积核计算出结果。

如下所示，我们实现一个[**计算多个通道的输出的互相关函数**]。


In [16]:
def corr2d_multi_in_out(X,K):
    """多输入通道和多输出通道的互相关计算"""
    # return torch.tensor([corr2d_multi_in(X, k) for k in K]) 
    #  stack 会保留计算图，但是 tensor 会直接创建新的张量
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [17]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [ ]:
corr2d_multi_in_out(X, K)


tensor([[[ 40.],
         [ 64.]],

        [[ 60.],
         [100.]],

        [[ 80.],
         [136.]]])

## $1 \times 1$ 卷积


$1 \times 1$卷积，即$k_h = k_w = 1$，看起来似乎没有多大意义。
毕竟，卷积的本质是有效提取相邻像素间的相关特征，而$1 \times 1$卷积显然没有此作用。
尽管如此，$1 \times 1$仍然十分流行，经常包含在复杂深层网络的设计中。下面，让我们详细地解读一下它的实际作用。

因为使用了最小窗口，$1\times 1$卷积失去了卷积层的特有能力——在高度和宽度维度上，识别相邻元素间相互作用的能力。
其实$1\times 1$卷积的唯一计算发生在通道上。

 :numref:`fig_conv_1x1`展示了使用$1\times 1$卷积核与$3$个输入通道和$2$个输出通道的互相关计算。
这里输入和输出具有相同的高度和宽度，输出中的每个元素都是从输入图像中同一位置的元素的线性组合。
我们可以将$1\times 1$卷积层看作在每个像素位置应用的全连接层，以$c_i$个输入值转换为$c_o$个输出值。
因为这仍然是一个卷积层，所以跨像素的权重是一致的。
同时，$1\times 1$卷积层需要的权重维度为$c_o\times c_i$，再额外加上一个偏置。

![互相关计算使用了具有3个输入通道和2个输出通道的 $1\times 1$ 卷积核。其中，输入和输出具有相同的高度和宽度。](../img/conv-1x1.svg)
:label:`fig_conv_1x1`